In [4]:
import pandas as pd
df = pd.read_csv( r'C:\c programming\churnguard_data.csv')
print("--- Dataset Shape ---")
print(df.shape)
# 1. Import pandas and load 'churnguard_data.csv' into a DataFrame
import pandas as pd

# Using your exact working file path
df = pd.read_csv(r'C:\c programming\churnguard_data.csv')

# 2. Print the shape of the dataset (rows, columns)
print("--- 2. Shape of the Dataset ---")
print(df.shape)
print()

# 3. Print the first 5 rows
print("--- 3. First 5 Rows ---")
print(df.head())
print()

# 4. Print column names and data types using .info()
print("--- 4. Column Names and Data Types ---")
df.info()
print()

# 5. Print the count of missing values in each column
print("--- 5. Count of Missing Values ---")
print(df.isnull().sum())
print()

# 6. Print the number of duplicate rows
print("--- 6. Number of Duplicate Rows ---")
print(df.duplicated().sum())
print()

# 7. Print the value counts of the 'Churn' column (to notice inconsistent entries)
print("--- 7. Value Counts of Churn Column ---")
print(df['Churn'].value_counts())
print()

# 8. Print the unique values in the 'Contract' column (to notice typos)
print("--- 8. Unique Values in Contract Column ---")
print(df['Contract'].unique())
print()


--- Dataset Shape ---
(1030, 12)
--- 2. Shape of the Dataset ---
(1030, 12)

--- 3. First 5 Rows ---
  c:\customerID  gender  SeniorCitizen  tenure PhoneService InternetService  \
0     CUST-0032    Male              0    21.0          YES     Fiber optic   
1     CUST-0110    Male              0    55.0          YES     Fiber optic   
2     CUST-0137  Female              1    46.0          Yes     Fiber optic   
3     CUST-0089  Female              1    63.0          Yes     Fiber optic   
4     CUST-0919  Female              0     8.0          Yes             DSl   

         Contract PaperlessBilling     PaymentMethod  MonthlyCharges  \
0  Month-to-month               No     Credit card             29.73   
1        Two year              Yes     Bank transfer           46.32   
2  Month-to-month               No    Mailed check             87.06   
3  Month-to-month              YES      Mailed check           56.97   
4  month to month               No  Electronic check           3

In [ ]:
MINI PROJECT TASK2 MODEL 2

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Load with encoding fix
df = pd.read_csv('churnguard_data.csv', encoding='utf-8-sig')

# Check column names first
print("Columns:", df.columns.tolist())

# Step 2: Drop customerID - safely
customer_col = [col for col in df.columns if 'customerID' in col or 'customer' in col.lower()]
if customer_col:
    df = df.drop(columns=customer_col)
    print("Dropped:", customer_col)

# Step 3: Remove duplicate rows
df = df.drop_duplicates()

# Step 4: Strip whitespace
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()

# Step 5: Standardise casing
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

# Step 6: Fix Contract
def fix_contract(value):
    if pd.isna(value):
        return value
    val = str(value).strip().lower()
    if val in ['month-to-month', 'monthly', 'month to month', 'm2m', 'mtm']:
        return 'Month-to-month'
    elif val in ['one year', '1 year', '1yr', 'one yr']:
        return 'One year'
    elif val in ['two year', '2 year', '2yr', 'two yr']:
        return 'Two year'
    else:
        return np.nan

df['Contract'] = df['Contract'].apply(fix_contract)

# Step 7: Fix InternetService
def fix_internet(value):
    if pd.isna(value):
        return value
    val = str(value).strip().lower()
    if val == 'dsl':
        return 'DSL'
    elif val in ['fiber optic', 'fibre optic', 'fiberoptic', 'fiber']:
        return 'Fiber optic'
    elif val in ['no', 'none']:
        return 'No'
    else:
        return np.nan

df['InternetService'] = df['InternetService'].apply(fix_internet)

# Step 8: Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 9: Remove tenure <= 0
df = df[df['tenure'] > 0]

# Step 10: Remove MonthlyCharges outliers
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]

# Step 11: Fill missing values
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(round(df['tenure'].median()))
df['tenure'] = df['tenure'].astype(int)

# Step 12: Shape
print("Cleaned DataFrame Shape:", df.shape)

# Step 13: Missing values
print("\nMissing Value Counts:")
print(df.isnull().sum())
# InternetService remaining NaN rows drop 
df = df.dropna(subset=['InternetService'])

print("Final Shape:", df.shape) 
print("\nMissing Value Counts:")
print(df.isnull().sum())

T

Columns: ['c:\\customerID', 'gender', 'SeniorCitizen', 'tenure', 'PhoneService', 'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Dropped: ['c:\\customerID']
Cleaned DataFrame Shape: (867, 11)

Missing Value Counts:
gender               0
SeniorCitizen        0
tenure               0
PhoneService         0
InternetService     14
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
dtype: int64
Final Shape: (853, 11)

Missing Value Counts:
gender              0
SeniorCitizen       0
tenure              0
PhoneService        0
InternetService     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


TASK 3 MINI PROJECT MODEL -2

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Load and Clean (Task 2 steps reused)
df = pd.read_csv('churnguard_data.csv', encoding='utf-8-sig')

customer_col = [col for col in df.columns if 'customerID' in col or 'customer' in col.lower()]
if customer_col:
    df = df.drop(columns=customer_col)

df = df.drop_duplicates()
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

def fix_contract(value):
    if pd.isna(value): return value
    val = str(value).strip().lower()
    if val in ['month-to-month','monthly','month to month']: return 'Month-to-month'
    elif val in ['one year','1 year']: return 'One year'
    elif val in ['two year','2 year']: return 'Two year'
    else: return np.nan

def fix_internet(value):
    if pd.isna(value): return value
    val = str(value).strip().lower()
    if val == 'dsl': return 'DSL'
    elif val in ['fiber optic','fibre optic','fiberoptic']: return 'Fiber optic'
    elif val in ['no','none']: return 'No'
    else: return np.nan

df['Contract'] = df['Contract'].apply(fix_contract)
df['InternetService'] = df['InternetService'].apply(fix_internet)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] > 0]
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(round(df['tenure'].median()))
df['tenure'] = df['tenure'].astype(int)
df = df.dropna(subset=['InternetService'])

# Step 2: Encode target column Churn
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Step 3: Encode categorical columns
cat_cols = ['gender', 'PhoneService', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Step 4: Separate X and y
X = df.drop(columns=['Churn'])
y = df['Churn']

# Step 5: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Step 6: Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 7: Accuracy Score
y_pred = model.predict(X_test)
print("Accuracy Score:", accuracy_score(y_test, y_pred))

# Step 8: Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Stay', 'Churn']))

model = LogisticRegression(max_iter=1000)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

Accuracy Score: 0.6549707602339181

Classification Report:
              precision    recall  f1-score   support

        Stay       0.64      0.96      0.77       102
       Churn       0.78      0.20      0.32        69

    accuracy                           0.65       171
   macro avg       0.71      0.58      0.55       171
weighted avg       0.70      0.65      0.59       171



c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


TASK -4 MINI PROJECT MODEL -2

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# =====================================================================
# Step 1: Load and Clean Dataset (Applying Task 2 cleaning steps)
# =====================================================================
# Load the dataset
df = pd.read_csv('churnguard_data.csv', encoding='utf-8-sig')

# Drop CustomerID if present
customer_col = [col for col in df.columns if 'customerID' in col.lower()]
if customer_col:
    df = df.drop(columns=customer_col)

# Drop duplicate rows
df = df.drop_duplicates()

# Standardize string formatting
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

# Data cleaning helper functions
def fix_contract(value):
    if pd.isna(value): return value
    val = str(value).strip().lower()
    if val in ['month-to-month', 'monthly', 'month to month']: return 'Month-to-month'
    elif val in ['one year', '1 year']: return 'One year'
    elif val in ['two year', '2 year']: return 'Two year'
    else: return np.nan

def fix_internet(value):
    if pd.isna(value): return value
    val = str(value).strip().lower()
    if val == 'dsl': return 'DSL'
    elif val in ['fiber optic', 'fibre optic', 'fiberoptic']: return 'Fiber optic'
    elif val in ['no', 'none']: return 'No'
    else: return np.nan

df['Contract'] = df['Contract'].apply(fix_contract)
df['InternetService'] = df['InternetService'].apply(fix_internet)

# Handle numerical values and ranges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] > 0]
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]

# Fill missing values with statistical defaults
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(round(df['tenure'].median()))
df['tenure'] = df['tenure'].astype(int)
df = df.dropna(subset=['InternetService'])

# =====================================================================
# Step 2: Target and Feature Encoding
# =====================================================================
# Map Churn into binary values
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Map Contract into ordinal values
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['Contract'] = df['Contract'].map(contract_map)

# =====================================================================
# Step 3: Retrain Logistic Regression Model on FULL Cleaned Dataset
# =====================================================================
# Extract required 5 features
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Contract']

df_model = df[features + ['Churn']].dropna()
X = df_model[features]
y = df_model['Churn']

# Fit model on entire available training data
model = LogisticRegression(max_iter=5000)
model.fit(X, y)

print("Model trained successfully on the full dataset!")
print("-" * 50)

# =====================================================================
# Step 4: Collect Interactive User Inputs via CLI
# =====================================================================
print("Enter customer details to predict churn:\n")

tenure = int(input("Enter tenure (months): "))
monthly = float(input("Enter Monthly Charges: "))
total = float(input("Enter Total Charges: "))
senior = int(input("Senior Citizen? (1 = Yes, 0 = No): "))
contract = int(input("Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): "))

# =====================================================================
# Step 5 & 6: Prediction Execution and Output Formatting
# =====================================================================
# Build temporary dataframe structure matching model requirements
user_input_df = pd.DataFrame([[tenure, monthly, total, senior, contract]], columns=features)

# Extract scalar outcome from list prediction
prediction = model.predict(user_input_df)[0]

print("\n" + "=" * 50)
if prediction == 1:
    print("Prediction: This customer is likely to CHURN.")
else:
    print("Prediction: This customer is likely to STAY.")
print("=" * 50)

Model trained successfully on the full dataset!
--------------------------------------------------
Enter customer details to predict churn:


Prediction: This customer is likely to CHURN.
